<a href="https://colab.research.google.com/github/pranavlandge36/RAG-Youtube-chatbot/blob/main/youtube_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [16]:
!pip install langchain langchain-openai dotenv youtube-transcript-api pytube pinecone openai tiktoken langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 16.1 MB/s eta 0:00:00


In [17]:
!pip install langchain-pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pinecone-plugin-assistant
    Found existing installation: pinecone-plugin-assistant 3.0.3
    Uninstalling pinecone-plugin-assistant-3.0.3:
      Successfully uninstalled pinecone-plugin-assistant-3.0.3
  Attempting uninstall: pinecone
    Found existing installation: pinecone 8.1.2
    Uninstalling pinecone-8.1.2:
      Successfully uninstalled

In [80]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from pinecone import Pinecone
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
video_id = "k-yDYgc8AmU" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

Learn drone programming using a Python-based simulator. This comprehensive course will teach you how to use Pyimverse, which is a highfidelity simulator where you can master AI drone programming using Python without the risk of expensive hardware crashes. You will progress from the fundamentals of 3D movement to deploying advanced computer vision across five practical missions, including gesture control and autonomous line following. Mortazza created this course. >> In many parts of the world, 2026 did not begin with fireworks. It began with thousands of synchronized drones lighting up the night sky. But drone shows are just the surface. Because these machines are already reshaping the real world. Agricultural drones spraying entire fields in minutes. Delivery drones flying across city skylines. Firefighting drones battling flames where humans can't reach. Police drones monitoring cities from above. Aerial advertisement screens floating in the sky. Even cars launching embedded drones d

In [4]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Learn drone programming using a', start=0.16, duration=4.4), FetchedTranscriptSnippet(text='Python-based simulator. This', start=2.159, duration=4.561), FetchedTranscriptSnippet(text='comprehensive course will teach you how', start=4.56, duration=4.56), FetchedTranscriptSnippet(text='to use Pyimverse, which is a', start=6.72, duration=4.4), FetchedTranscriptSnippet(text='highfidelity simulator where you can', start=9.12, duration=5.84), FetchedTranscriptSnippet(text='master AI drone programming using Python', start=11.12, duration=6.159), FetchedTranscriptSnippet(text='without the risk of expensive hardware', start=14.96, duration=4.88), FetchedTranscriptSnippet(text='crashes. You will progress from the', start=17.279, duration=4.961), FetchedTranscriptSnippet(text='fundamentals of 3D movement to deploying', start=19.84, duration=5.279), FetchedTranscriptSnippet(text='advanced computer vision across five', start=22.24, duration

# Text Splitting

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

In [6]:
chunks=splitter.create_documents([transcript])
chunks

[Document(metadata={}, page_content="Learn drone programming using a Python-based simulator. This comprehensive course will teach you how to use Pyimverse, which is a highfidelity simulator where you can master AI drone programming using Python without the risk of expensive hardware crashes. You will progress from the fundamentals of 3D movement to deploying advanced computer vision across five practical missions, including gesture control and autonomous line following. Mortazza created this course. >> In many parts of the world, 2026 did not begin with fireworks. It began with thousands of synchronized drones lighting up the night sky. But drone shows are just the surface. Because these machines are already reshaping the real world. Agricultural drones spraying entire fields in minutes. Delivery drones flying across city skylines. Firefighting drones battling flames where humans can't reach. Police drones monitoring cities from above. Aerial advertisement screens floating in the sky. 

In [7]:
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-ada-002",  # or any model from OpenRouter
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    # streaming=True
)

In [13]:
from pinecone import Pinecone, ServerlessSpec

In [15]:
# DONT RUN THIS TWICE-> IT WILL CRASH AS INDEX ALREADY EXISTS
pc=Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "youtube-chatbot"

existing_indexes = [i.name for i in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

In [10]:
# vectors=embeddings.embed_query('how are you')
# len(vectors)

1536

In [16]:
index

In [19]:
from langchain_core import documents
vector_store=PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name
)

In [20]:
vector_store

In [23]:
results = index.fetch(
    ids=["2ceefbbf-19d6-4dda-8b13-7feeb1d142c2"]
)

print(results)

FetchResponse(namespace='', vectors={'2ceefbbf-19d6-4dda-8b13-7feeb1d142c2': Vector(id='2ceefbbf-19d6-4dda-8b13-7feeb1d142c2', values=[-0.0128188152, 0.0169497523, -0.0293038376, 0.00204449124, -0.0184730366, -0.0103337979, -0.0104757994, -0.0247727148, -0.0039276178, -0.0305689368, 0.0205126852, 0.0364297032, -0.0136191845, -0.00641263509, -0.00375979859, 0.00916551705, 0.00639327103, -0.000980290817, 0.00708068488, -0.0108888932, 0.010843711, 0.00795205496, -0.0383144431, -0.0260636341, -0.0258570872, 0.00365329767, 0.0210032351, -0.0235076156, 0.0150908306, 0.00273513235, 0.00586076733, 0.00369847985, 0.00543799205, -0.0328925885, -0.0184472166, 0.0129349977, 0.0129737249, -0.0116376253, 0.00136998668, -0.0175693929, 0.0330475, 0.0263992716, -0.00394698139, 0.0183310341, -0.00112229178, -0.0249276254, 0.0101788882, -0.0378238969, -0.00688704709, 0.00917842612, -0.0081198737, 0.0180212148, -0.0475574173, -0.0321180373, 0.0274965521, 0.0286583789, 0.0248888973, 0.0138257314, -0.000220

In [25]:
retriever=vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 5})
retriever

VectorStoreRetriever(tags=['PineconeVectorStore', 'OpenAIEmbeddings'], vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x7af4bc5c1640>, search_kwargs={'k': 5})

In [26]:
retriever.invoke('what is pysimverse')

[Document(id='224fb1db-60f9-4dd0-bf2a-41c280138089', metadata={}, page_content='of stability and lifting capacity depending on its purpose. Have a look at the drone components. A frame that holds all components together. Motors that generate lift for flight. Propellers that create thrust by spinning. An ESC electronic speed controller that controls motor speed. Power distribution board that distributes battery power. Flight controller that is the brain of the drone. Battery that supplies electrical power. Receiver or Bluetooth that receives control signals. Camera that captures photos and videos. Video transmitter that sends video wirelessly. Sensors that measure movement and position. A barometer that measures altitude. a GPS that tracks drone location and an IMU that measures angle and acceleration. So the first thing we will need is PIMverse. So we will head to pyimburse.com. And what exactly is piverse? It is a simulator that has a lots uh that has lots of missions where you can pr

In [44]:
!pip install langchain-nvidia-ai-endpoints

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 2.3 MB/s eta 0:00:00


In [107]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
model = ChatNVIDIA(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("NVIDIA_API_KEY"),

)

In [108]:
prompt=PromptTemplate(
    template="""
    You are a helpful Youtube assistant, you will be provided youtube video transcripts.
    Answer only from transcripts.
    If information not present in transcript , just say i dont know.
    {context}
    Question:{question}
    """,
    input_variables=['context','question']

)

In [114]:
question='what is Pyimverse?'
retrieved_docs=retriever.invoke(question)

In [115]:
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)

In [116]:
final_prompt = prompt.invoke({
    "context": context_text,
    "question": question
})

In [105]:
final_prompt

StringPromptValue(text="\n    You are a helpful Youtube assistant, you will be provided youtube video transcripts.\n    Answer only from transcripts.\n    If information not present in transcript , just say i dont know.\n    I will Can we make it smaller? No. Okay. But I will put it here. So, I need to jump. Jump. Jump. There you go. And now I need to jump again. Jump. That's a lot of exercise. And uh it will become faster and faster. So, you have to be careful. jump. Okay. Okay. Uh there you go. I was able to cross only two. Okay. So that was how you can actually uh play this game. It's a lot of fun to actually do that and it is quite tiring as well. So you have to do a lots of jumps. So later on there will be lots of more games and examples like this. So right now it's uh just uh two of these games and the other ones are mostly educational but uh in future there will be a lot more uh scenes and missions where you will be able to do lots of these different uh games and uh other stuff.

In [117]:
answer=model.invoke(final_prompt)

In [119]:
answer.content

'Pyimverse is a Python‑based, high‑fidelity drone simulator. It provides a wide range of missions where you can program and test drones in a virtual environment, allowing you to practice AI drone programming with Python without needing real hardware.'

In [120]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [121]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [122]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [123]:
parallel_chain.invoke('who is Pyimverse')

{'context': "of stability and lifting capacity depending on its purpose. Have a look at the drone components. A frame that holds all components together. Motors that generate lift for flight. Propellers that create thrust by spinning. An ESC electronic speed controller that controls motor speed. Power distribution board that distributes battery power. Flight controller that is the brain of the drone. Battery that supplies electrical power. Receiver or Bluetooth that receives control signals. Camera that captures photos and videos. Video transmitter that sends video wirelessly. Sensors that measure movement and position. A barometer that measures altitude. a GPS that tracks drone location and an IMU that measures angle and acceleration. So the first thing we will need is PIMverse. So we will head to pyimburse.com. And what exactly is piverse? It is a simulator that has a lots uh that has lots of missions where you can program your drone to execute those missions. So for example here you

In [124]:
parser = StrOutputParser()

In [125]:
main_chain = parallel_chain | prompt | model | parser

In [126]:
main_chain.invoke('summarize this video')

'**Video Summary**\n\n- The creator explains a new AI‑driven workflow for coding drone‑simulation projects: first build reusable **templates**, then generate full scripts from those templates.  \n- This method is the core of a **complete course** the speaker has developed, which walks viewers through many real‑world‑style missions (rescue, agriculture, drone shows, etc.).  \n- Viewers can download the associated files (the speaker plans to upload them for easy access) and try the code themselves.  \n- A key demo shows how to capture a screenshot in the simulator: pressing **Z** saves the current video frame to a new folder with a timestamped filename.  \n- The simulator’s keyboard controls are described (arrow keys for forward/left/right; up/down arrows now control altitude).  \n- The project is being launched via a **Kickstarter campaign**. Backers receive **lifetime, one‑time‑payment access** to all current and future missions and levels.  \n- The free offering includes **six beginne